# 04 — Creating Basic Tasks and DAGs

> **DataMindAI | Ahmed**

## What is a DAG?
A **DAG (Directed Acyclic Graph)** is a diagram of tasks where arrows show dependencies.
Lakeflow Jobs builds the DAG visually as you add tasks.

## The scenario
Three tasks where B and C run **in parallel** only after A succeeds:
```
Task A: ingest_bronze
    │
    ├──────────────┐
    ▼              ▼
Task B:         Task C:
transform_silver  validate_quality
```

## Dependency conditions
| Condition | Meaning |
|-----------|---------|
| `ALL_SUCCESS` | Run only if ALL upstream tasks succeeded |
| `ALL_FAILED` | Run only if ALL upstream tasks failed |
| `ALL_DONE` | Run regardless of upstream result |
| `AT_LEAST_ONE_SUCCESS` | Run if at least one upstream succeeded |

---
## Task A — ingest_bronze
---

### Job UI Setup
1. Create job: **+ Create Job** → name: `Student Pipeline DAG Demo`
2. First task: name `ingest_bronze` | Type: Notebook | Compute: Serverless
3. Add Retry: Max retries: 1 | Retry wait: 15 seconds

In [ ]:
# ── TASK A: ingest_bronze ─────────────────────────────────────────────────────
# This is Task A in the DAG. Runs first, with no dependencies.
# Retry policy: 1 retry, 15-second wait (configured in Job UI)

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit
import datetime

run_date = datetime.date.today().isoformat()
print(f'[Task A] ingest_bronze starting — {run_date}')

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94),
]

df = (spark.createDataFrame(students)
        .withColumn('run_date', lit(run_date))
        .withColumn('ingested_at', current_timestamp()))

df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.students_raw')

count = df.count()
print(f'[Task A] SUCCESS — {count} records written to bronze.students_raw')

# Publish for downstream tasks
dbutils.jobs.taskValues.set(key='record_count', value=int(count))
dbutils.jobs.taskValues.set(key='run_date',     value=run_date)

---
## Task B — transform_silver (depends on A)
---

### Job UI Setup
1. Click **+** on the canvas → Add task
2. Name: `transform_silver` | Depends on: `ingest_bronze` | Run if: **ALL_SUCCESS**
3. Notification: on failure → your email

In [ ]:
# ── TASK B: transform_silver ─────────────────────────────────────────────────
# Runs only if Task A succeeded (ALL_SUCCESS dependency)
# Publishes pass/fail counts as task values

from pyspark.sql.functions import when, col

print('[Task B] transform_silver starting')

df = spark.table('demo_catalog.bronze.students_raw')

df_silver = df.withColumn('grade',
    when(col('score') >= 70, 'Distinction')
    .when(col('score') >= 55, 'Merit')
    .when(col('score') >= 40, 'Pass')
    .otherwise('Fail')
).withColumn('at_risk',
    when((col('score') < 40) | (col('attendance') < 50), True).otherwise(False)
)

df_silver.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.silver.students_graded')

pass_count = df_silver.filter("grade != 'Fail'").count()
fail_count = df_silver.filter("grade == 'Fail'").count()
at_risk    = df_silver.filter('at_risk = true').count()

print(f'[Task B] SUCCESS — Pass: {pass_count}, Fail: {fail_count}, At-risk: {at_risk}')
df_silver.select('name','score','grade','at_risk').show()

dbutils.jobs.taskValues.set(key='pass_count', value=int(pass_count))
dbutils.jobs.taskValues.set(key='fail_count', value=int(fail_count))
dbutils.jobs.taskValues.set(key='at_risk',    value=int(at_risk))

---
## Task C — validate_quality (runs in parallel with B)
---

### Job UI Setup
1. Add another task: name `validate_quality`
2. **Depends on: `ingest_bronze`** (same parent as Task B → runs in parallel)
3. Run if: ALL_SUCCESS

> **Key point:** Both B and C depend on A, so they start simultaneously after A finishes.

In [ ]:
# ── TASK C: validate_quality (runs in parallel with Task B) ──────────────────
# Checks data quality rules on the Bronze data

print('[Task C] validate_quality starting (parallel with Task B)')

df = spark.table('demo_catalog.bronze.students_raw')

issues = []

# Rule 1: No null student IDs
null_ids = df.filter('student_id IS NULL').count()
if null_ids > 0:
    issues.append(f'NULL student_ids: {null_ids}')

# Rule 2: Scores must be 0-100
bad_scores = df.filter('score < 0 OR score > 100').count()
if bad_scores > 0:
    issues.append(f'Invalid scores (out of 0-100): {bad_scores}')

# Rule 3: Attendance must be 0-100
bad_att = df.filter('attendance < 0 OR attendance > 100').count()
if bad_att > 0:
    issues.append(f'Invalid attendance: {bad_att}')

# Rule 4: Expected row count
count = df.count()
if count == 0:
    issues.append('CRITICAL: Zero records in bronze table')

if issues:
    print('[Task C] QUALITY ISSUES FOUND:')
    for issue in issues:
        print(f'  ⚠ {issue}')
else:
    print(f'[Task C] All {count} records passed quality checks ✓')

quality_status = 'PASS' if not issues else 'FAIL'
dbutils.jobs.taskValues.set(key='quality_status', value=quality_status)
dbutils.jobs.taskValues.set(key='quality_issues',  value=len(issues))

---
## Error Handler — ALL_FAILED dependency
---

### Job UI Setup
1. Add task: `error_notifier`
2. Depends on: `ingest_bronze` | Run if: **ALL_FAILED**
3. This task only executes when Task A fails

> **Demo tip:** Add `raise Exception('test')` to Task A, run, see error_notifier fire.

In [ ]:
# ── ERROR HANDLER: runs only when ingest_bronze FAILS ────────────────────────
# Run if: ALL_FAILED (configured in Job UI dependency condition)
import datetime

print('=' * 55)
print('PIPELINE FAILURE ALERT')
print(f'Time: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('Task: ingest_bronze')
print('Action: On-call team has been notified.')
print('Next steps: Check Run History tab for stack trace.')
print('=' * 55)

# Write to audit log
from pyspark.sql import Row
spark.createDataFrame([Row(
    event='ingest_failure',
    detected_at=datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    task='ingest_bronze'
)]).write.format('delta').mode('append').saveAsTable('demo_catalog.config.pipeline_audit')